In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, label_binarize, MaxAbsScaler
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score, roc_curve, PrecisionRecallDisplay
import seaborn as sns
from global_land_mask import globe
import rasterio
import rasterio.features
import rasterio.warp
import h3

SEED = 0
LABEL_SMOOTHING = 0.0

In [6]:
SEED=0
res=4
Abs_per_presence=1

# Load training data
train = np.load('species/species_train.npz')

# Grid- every H3 cell on the globe
grid_cells = [c for r0 in h3.get_res0_cells() for c in h3.cell_to_children(r0, res)]
grid = np.array([h3.cell_to_latlng(c) for c in grid_cells]) # centers
cell_index = {c: i for i, c in enumerate(grid_cells)}
print('Grid cells-', len(grid))

df = pd.DataFrame({'lat': train['train_locs'][:, 0], 'lon': train['train_locs'][:, 1], 'species': train['train_ids'],})
print('Sightings-', len(df))

def near_land(lat, lon, step=.25): # .25 d- about 28km N-S
    near = np.zeros(len(lat), dtype=bool)
    for dlat in (-step, 0, step):
        for dlon in (-step, 0, step):
            clip_lat = np.clip(lat+dlat,-90,90)
            clip_long = np.clip(lon+dlon,-180,180)
            near |= globe.is_land(clip_lat, clip_long)
    return near

# Remove ocean sightings- about 25km from land
df = df[near_land(df['lat'].values, df['lon'].values)].copy()
print('Total after removing ocean sightings-', len(df))
land_cells = np.flatnonzero(near_land(grid[:, 0], grid[:, 1]))
print('Land cells-', len(land_cells))

# Bin- find the H3 cell each sighting falls in
df['cell'] = [cell_index[h3.latlng_to_cell(lat, lon, res)] for lat, lon in zip(df['lat'], df['lon'])]

# Index for species 0-499
species_index = {taxon: i for i, taxon in enumerate(train['taxon_ids'])}
df['index'] = df['species'].map(species_index)

# Presence only- one row per cell, list of species seen there
presences = df.groupby('cell')['index'].unique().apply(sorted).reset_index(name='species')
presences['n_sightings'] = df.groupby('cell').size().values
print('Cells with sightings-', len(presences))

# Pseudo-absences- per species
rng = np.random.default_rng(SEED)
pairs = df[['cell', 'species']].drop_duplicates()  # one row per (cell, species) presence
absent_cells = []
absent_species = []
for species, group in pairs.groupby('species'):
    candidates = np.setdiff1d(land_cells, group['cell'])  # Pseudo-absences in cells where presences are not there
    picked = rng.choice(candidates, len(group)*Abs_per_presence, replace=False)
    absent_cells.extend(picked)
    absent_species.extend([species_index[species]] * len(picked))

# One row per cell, with the list of species pseudo-absence
absences = pd.DataFrame({'cell': absent_cells, 'absent': absent_species})
absences = absences.groupby('cell')['absent'].apply(list).reset_index()
absences['species'] = [[] for _ in range(len(absences))]  # no species seen
absences['n_sightings'] = 0

data = pd.concat([presences, absences], ignore_index=True)
data['absent'] = [a if type(a) == list else [] for a in data['absent']]
data['n_species'] = data['species'].apply(len)
print('Pseudo-absences-', len(absent_cells)) # Number of pseudo absences

# Taxonomy class for species
taxonomy = pd.read_csv('species/species_taxonomy.csv')
species_class = dict(zip(taxonomy['taxon_id'], taxonomy['class']))

# Number for each class
# {0: 'Amphibia', 1: 'Aves', 2: 'Crocodylia', 3: 'Mammalia', 4: 'Squamata', 5: 'Testudines'}
class_names = sorted(taxonomy['class'].unique())
class_index = {}
for i, c in enumerate(class_names):
    class_index[c] = i
class_id = []
for t in train['taxon_ids']:
    class_id.append(class_index[species_class[t]])
class_id = np.array(class_id)

# Features- normalise lat/lon to [-1,1], then sin/cos (SINR paper approach)
def encode(latlon):
    lat = latlon[:,0]/90
    lon = latlon[:,1]/180
    return np.column_stack([np.sin(np.pi*lon), np.cos(np.pi*lon), np.sin(np.pi*lat), np.cos(np.pi*lat)])

X_train = encode(grid[data['cell']])
y_train = np.zeros((len(data), len(species_index)), dtype=bool) # 1=species present
for row, species in enumerate(data['species']):
    y_train[row, species] = 1

absent_train = np.zeros((len(data), len(species_index)), dtype=bool)
for row, absent in enumerate(data['absent']):
    absent_train[row, absent] = 1

# Save
np.savez('species/preprocessed_cells.npz', X_train=X_train, y_train=y_train, absent_train=absent_train, class_id=class_id, n_sightings=data['n_sightings'].values, n_species=data['n_species'].values, cell_train=data['cell'].values, grid_latlon=grid, grid_cells=np.array(grid_cells), res=res, taxon_ids=train['taxon_ids'])
print('X_train-', X_train.shape, ', y_train-', y_train.shape)

# Usage for models
preprocessed = np.load('species/preprocessed_cells.npz')
cols = ['sin_lon', 'cos_lon', 'sin_lat', 'cos_lat']
train_df = pd.DataFrame(preprocessed['X_train'], columns=cols)
train_df['cell'] = preprocessed['cell_train']
train_df['n_species'] = preprocessed['n_species']
train_df['sightings'] = preprocessed['n_sightings']
train_df['species'] = [np.flatnonzero(row) for row in preprocessed['y_train']] # list view

X = preprocessed['X_train']
y = preprocessed['y_train'].astype(np.float32) # for NN, 0/1 col for each grid cell for all 500 species

# NaN 1=present, 0=absent, NaN=unknown 
y_nan = y.copy()
y_nan[(y_nan == 0) & ~preprocessed['absent_train']] = np.nan
print('present-', (y_nan == 1).sum(), 'absent-', (y_nan == 0).sum(), 'unknown-', np.isnan(y_nan).sum())

print(train_df.head())
print(f'X shape = {X.shape}')
print(f'y shape = {y_nan.shape}')
print(X[2], y_nan[2])

Grid cells- 288122
Sightings- 272037
Total after removing ocean sightings- 271239
Land cells- 89277
Cells with sightings- 16919
Pseudo-absences- 69497
X_train- (65215, 4) , y_train- (65215, 500)
present- 69497 absent- 69497 unknown- 32468506
    sin_lon   cos_lon   sin_lat   cos_lat  cell  n_species  sightings  \
0  0.517536  0.855661  0.632448 -0.774602  1671          1          1   
1  0.491174  0.871061  0.637845 -0.770165  1701          1          1   
2  0.474143  0.880448  0.636310 -0.771434  1706          4         26   
3 -0.866532 -0.499122  0.554682 -0.832062  3171          1          1   
4 -0.966489 -0.256710  0.667645 -0.744480  3499          1          1   

              species  
0               [111]  
1               [111]  
2  [25, 60, 227, 446]  
3               [437]  
4               [437]  
X shape = (65215, 4)
y shape = (65215, 500)
[ 0.47414342  0.88044762  0.63630974 -0.77143368] [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan

In [9]:
data = np.load('./Provided Code/species_train.npz')
train_locs = data['train_locs']  # 2D array, rows are number of datapoints and 
                                 # columns are "latitude" and "longitude"
train_ids = data['train_ids']    # 1D array, entries are the ID of the species 
                                 # that is present at the corresponding location in train_locs
species = data['taxon_ids']      # list of species IDe. Note these do not necessarily start at 0 (or 1)
species_names = dict(zip(data['taxon_ids'], data['taxon_names']))  # latin names of species 

# loading test data 
data_test = np.load('./Provided Code/species_test.npz', allow_pickle=True)
test_locs = data_test['test_locs']    # 2D array, rows are number of datapoints 
                                      # and columns are "latitude" and "longitude"
# data_test['test_pos_inds'] is a list of lists, where each list corresponds to 
# the indices in test_locs where a given species is present, it can be assumed 
# that they are not present in the other locations 
test_pos_inds = dict(zip(data_test['taxon_ids'], data_test['test_pos_inds'])) 

# data stats
print('Train Stats:')
print('Number of species in train set:           ', len(species))
print('Number of train locations:                ', train_locs.shape[0])
_, species_counts = np.unique(train_ids, return_counts=True)
print('Average number of locations per species:  ', species_counts.mean())
print('Minimum number of locations for a species:', species_counts.min())
print('Maximum number of locations for a species:', species_counts.max())

Train Stats:
Number of species in train set:            500
Number of train locations:                 272037
Average number of locations per species:   544.074
Minimum number of locations for a species: 50
Maximum number of locations for a species: 2000


In [10]:
X = train_locs.copy()
y = train_ids.copy()

In [11]:
temp_loc = pd.DataFrame(data["train_locs"])
temp_loc = temp_loc.rename(columns={0:"Lat",1:"Lon"})
temp_loc["train_ids"] = data["train_ids"]
temp_loc

,Lat,Lon,train_ids
0,-18.286728,143.481247,31529
1,-13.099798,130.783646,31529
2,-13.965274,131.695145,31529
3,-12.853950,132.800507,31529
4,-12.196790,134.279327,31529
...,...,...,...
272032,33.716885,73.203621,145031
272033,24.600239,72.730560,145031
272034,18.849600,80.654129,145031
272035,21.073837,75.945656,145031


In [12]:
preprocessed = np.load('./PreprocessData/preprocessed_data.npz')
cols = ['sin_lon', 'cos_lon', 'sin_lat', 'cos_lat']
train_df = pd.DataFrame(preprocessed['X_train'], columns=cols)
train_df['species'] = preprocessed['species_train']
train_df['present'] = preprocessed['y_train']
test_df = pd.DataFrame(preprocessed['X_test'], columns=cols)
train_df = train_df[train_df.groupby("species").transform('size') > 50].reset_index(drop=True)

# For model inputs
cols_train = ['sin_lon', 'cos_lon', 'sin_lat', 'cos_lat', 'species']
cols_train_2 = ['sin_lon', 'cos_lon', 'sin_lat', 'cos_lat', 'present']
X_pp = train_df[cols_train].values
X_2_pp = train_df[cols_train_2].values
y_pp = train_df['present'].values
y_2_pp = train_df['species'].values
X_train_pp, X_val_pp, y_train_pp, y_val_pp = train_test_split(X_pp, y_pp, test_size=0.25, random_state=SEED, stratify=X_pp[:,4])
#X_test_pp, X_val_pp, y_test_pp, y_val_pp = train_test_split(X_test_pp, y_test_pp, test_size=0.25, random_state=SEED, stratify=X_test_pp[:,4])
X_train_2_pp, X_test_2_pp, y_train_2_pp, y_test_2_pp = train_test_split(X_2_pp, y_2_pp, test_size=0.33, random_state=SEED, stratify=y_2_pp)
#X_train_list.append(X_train_pp)
#y_train_list.append(y_train_pp)
#X_train_list.append(X_train_2_pp)
#y_train_list.append(y_train_2_pp)

In [13]:
print(len(np.unique(y_test_2_pp)))
print(len(np.unique(y_train_2_pp)))

420
420


In [14]:
rfc2 = RandomForestClassifier(criterion="entropy")
rfc2 = rfc2.fit(X_train_2_pp, y_train_2_pp)

MemoryError: could not allocate 440401920 bytes

In [ ]:
pred_2_pp = rfc2.predict(X_test_2_pp)
acc_t_rfc2 = rfc2.score(X_train_2_pp, y_train_2_pp)
acc_rfc2 = rfc2.score(X_test_2_pp, y_test_2_pp)
matrix_2 = confusion_matrix(y_test_2_pp, pred_2_pp)
matrix_2 = matrix_2.astype('float') / matrix_2.sum(axis=1)[:, np.newaxis]
print(f"{'='*10}\nTrain: {acc_t_rfc2*100:.2f}%\nTest: {acc_rfc2*100:.2f}%")
pred_prob_2 = rfc2.predict_proba(X_test_2_pp)

In [ ]:
proba_2 = rfc2.predict_proba(X_test_2_pp)[:, 1]
classes = rfc2.classes_

Y_bin = label_binarize(y_test_2_pp, classes=classes)

aps, aucs = {}, {}
for k, sp in enumerate(classes):
    pos = Y_bin[:, k].sum()
    aps[sp] = average_precision_score(Y_bin[:, k], proba[:, k])

mAP = 100 * np.mean(list(aps.values()))
mAUC = roc_auc_score(y_test_2_pp, proba, multi_class="ovr", average="macro")

print(f"mAP: {mAP}\nmAUC: {mAUC}")

NameError: name 'rfc2' is not defined

In [ ]:
sns.heatmap(matrix_2, annot=True, fmt=".2f")
plt.xticks(rotation=25)
plt.yticks(rotation=0)
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

In [ ]:
roc_auc = roc_auc_score(y_test_2_pp, proba_2)
fpr, tpr, _ = roc_curve(y_test_2_pp, proba_2)

fig, ax = plt.subplots()
ax.axline((0, 0), slope=1, color="black", linestyle=(0, (5, 5)))
ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.2f}")
plt.xlim([-0.01,1.01])
plt.ylim([-0.01,1.01])
plt.legend()
plt.title("ROC_AUC")
plt.show()